# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Operating Model

1. Mount Drive from the Colab Files sidebar if `/content/drive/MyDrive` is not present.
2. Run the bootstrap shell cell.
3. Optionally list profiles.
4. Use the profile shell cell as a terminal command launcher.
5. Inspect the debug status cell if a profile fails; debug bundles are Drive-backed.

Default profile is `full-eval-dry-run` because full preprocessing is assumed complete. Full eval now has strict readiness gates: raw WOMD must be complete/readable, and full preprocessing must have `_SUCCESS` plus `preprocess_manifest.json`. If GCS reads fail with anonymous access, run `stage-full-womd-validation` once, then rerun eval with the local Drive-bound raw root `/content/latentdriver-waymax-experiments/artifacts/assets/raw_womd`.


In [ ]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Default: lightweight readiness check. This should pass before full evaluation.
python3 scripts/colab_canary.py \
  --profile full-eval-dry-run \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --vis video \
  --auto-install-runtime \
  --waymo-dataset-root /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd

# If the dry-run reports missing raw WOMD shards, stage all 150 validation shards once.
# This is resumable and skips existing non-empty shards. It can take a long time.
# python3 scripts/colab_canary.py \
#   --profile stage-full-womd-validation \
#   --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0

# After raw WOMD staging and full preprocessing are complete, run a real single-model full eval.
# python3 scripts/colab_canary.py \
#   --profile full-eval-reactive-single \
#   --model latentdriver_t2_j3 \
#   --seed 0 \
#   --vis video \
#   --auto-install-runtime \
#   --waymo-dataset-root /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

printf '\n--- debug root ---\n'
ls -lah results/debug_runs || true

printf '\n--- current latest run: LATEST.json ---\n'
cat results/debug_runs/LATEST.json || true

printf '\n--- historical latest failure: LATEST_FAILURE.json ---\n'
printf 'This is only the most recent failed run, not necessarily the current run. Trust LATEST.json for current status.\n'
if [ -f results/debug_runs/LATEST_FAILURE.json ]; then
  cat results/debug_runs/LATEST_FAILURE.json
else
  printf 'No failure pointer recorded yet.\n'
fi

printf '\n--- stable aliases ---\n'
find results/debug_runs -maxdepth 2 -type f \
  \( -name 'ALIAS.json' -o -name 'manifest.json' -o -name 'failure_summary.json' \) \
  -print | sort || true
